In [ ]:
import sys
import os
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import pandas as pd
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.torch_lightning.lightning_data_module import DroneDataModule
from src.torch_lightning.lightning_module import DroneClassifier

pl.seed_everything(42)

In [ ]:
N_MELS = 128

dm = DroneDataModule(
    root_dir="data",
    dataset_type="binary",
    batch_size=16,
    num_workers=0,
    n_mels=N_MELS
)

dm.prepare_data()
dm.setup()

num_classes = len(dm.class_to_idx)
print(f"Liczba wykrytych klas: {num_classes} ({dm.class_to_idx})")

In [ ]:
model = DroneClassifier(
    num_classes=num_classes,
    n_mels=N_MELS,
    learning_rate=1e-4
)

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    dirpath="checkpoints",
    filename="drone-mamba-{epoch:02d}-{val_acc:.2f}",
    save_top_k=1,
    mode="max",
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min"
)

logger = CSVLogger("logs", name="drone_detection_mamba")

In [ ]:
trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=logger,
    log_every_n_steps=5
)

trainer.fit(model, datamodule=dm)

In [ ]:
trainer.test(model, datamodule=dm, ckpt_path="best")

In [ ]:
metrics_path = os.path.join(logger.log_dir, "metrics.csv")
if os.path.exists(metrics_path):
    metrics = pd.read_csv(metrics_path)
    
    train_loss = metrics.dropna(subset=['train_loss'])
    val_loss = metrics.dropna(subset=['val_loss'])

    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_loss['epoch'], train_loss['train_loss'], label='Train Loss')
    plt.plot(val_loss['epoch'], val_loss['val_loss'], label='Val Loss')
    plt.xlabel('Epoka')
    plt.title('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(val_loss['epoch'], val_loss['val_acc'], label='Val Acc', color='orange')
    plt.xlabel('Epoka')
    plt.title('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.show()